<div dir="rtl">

# کارگاه جلسه ۲۱ — بیز ساده و رگرسیون لجستیک

هر بخش مستقل است. نوشتن این کدها در امتحان پرسیده نمی‌شود.

1. تمرین الف: تشخیص پیامک هرزنامه با بیز ساده (۵۵۷۲ پیامک واقعی)
2. تمرین ب: تنظیم C و تفسیر نسبت شانس در مدل کامل تایتانیک

</div>

<div dir="rtl">

## تمرین الف — پیامک هرزنامه (۱۰ دقیقه)

1. داده را از `https://raw.githubusercontent.com/justmarkham/DAT8/master/data/sms.tsv` (جداشده با tab، بدون سرستون؛ ستون اول برچسب و دوم متن) بارگذاری کنید. چند درصد پیامک‌ها هرزنامه‌اند؟
2. داده را با ۲۰ درصد آزمون (طبقه‌بندی‌شده، `random_state=42`) تقسیم کنید.
3. دو Pipeline بسازید: `CountVectorizer` و `MultinomialNB`؛ `CountVectorizer(binary=True)` و `BernoulliNB`. دقت اعتبارسنجی متقابل ۵ بخشی و دقت آزمون هر دو را با مدل پایه مقایسه کنید.
4. ده کلمه‌ای که بیشترین نسبت احتمال هرزنامه به عادی را دارند پیدا کنید.
5. چرا `CountVectorizer` باید داخل Pipeline باشد؟

</div>

In [1]:
# Write your solution here

<div dir="rtl">

### پاسخ تمرین الف

</div>

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB, BernoulliNB
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyClassifier

sms = pd.read_csv("https://raw.githubusercontent.com/justmarkham/DAT8/master/data/sms.tsv",
                  sep="\t", header=None, names=["label", "text"])
print(f"messages={len(sms)}  spam share={(sms['label'] == 'spam').mean():.3f}")
X_train, X_test, y_train, y_test = train_test_split(sms["text"], sms["label"], test_size=0.2,
                                                    stratify=sms["label"], random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
models = {"Baseline": DummyClassifier(strategy="most_frequent"),
          "Multinomial NB": make_pipeline(CountVectorizer(), MultinomialNB()),
          "Bernoulli NB": make_pipeline(CountVectorizer(binary=True), BernoulliNB())}
for name, model in models.items():
    cv_acc = cross_val_score(model, X_train, y_train, cv=cv).mean()
    test_acc = model.fit(X_train, y_train).score(X_test, y_test)
    print(f"{name:15s} CV accuracy={cv_acc:.4f}  test accuracy={test_acc:.4f}")

mnb = models["Multinomial NB"]
vec, nb = mnb[0], mnb[-1]
logp = pd.DataFrame(nb.feature_log_prob_, index=nb.classes_, columns=vec.get_feature_names_out())
ratio = (logp.loc["spam"] - logp.loc["ham"]).sort_values(ascending=False)
print("most spammy words:", list(ratio.index[:10]))

messages=5572  spam share=0.134
Baseline        CV accuracy=0.8658  test accuracy=0.8664


Multinomial NB  CV accuracy=0.9863  test accuracy=0.9874


Bernoulli NB    CV accuracy=0.9767  test accuracy=0.9758
most spammy words: ['claim', 'prize', '150p', 'tone', '18', 'cs', '500', 'guaranteed', '1000', 'uk']


<div dir="rtl">

### پاسخ‌های کلیدی الف

- داده ۵۵۷۲ پیامک دارد که **۱۳.۴ درصد** آن‌ها هرزنامه است؛ مدل پایه «همیشه عادی» دقت حدود ۰.۸۶۶ دارد.
- بیز چندجمله‌ای: دقت اعتبارسنجی **۰.۹۸۶** و دقت آزمون **۰.۹۸۷**. بیز برنولی: ۰.۹۷۷ و ۰.۹۷۶. هر دو بسیار بهتر از مدل پایه‌اند؛ برای پیامک، شمارش تکرار کمی بهتر از حضور یا نبود کلمه است.
- کلمه‌های هرزنامه‌ای: claim، prize، 150p، tone، 18، cs، 500، guaranteed، 1000 و uk؛ یعنی جایزه، مبلغ و شماره‌های تماس پولی.
- واژگان `CountVectorizer` از داده **یاد گرفته می‌شود** (گام یادگیرنده)؛ اگر بیرون از Pipeline روی کل داده fit شود، کلمه‌های بخش اعتبارسنجی و آزمون هم در واژگان می‌آیند.

</div>

<div dir="rtl">

## تمرین ب — مدل کامل تایتانیک (۱۵ دقیقه)

1. با ویژگی‌ها، تقسیم و پیش‌پردازش استاندارد جلسه ۲۰، پارامتر C رگرسیون لجستیک را از میان ۰.۰۱، ۰.۱، ۱، ۱۰ و ۱۰۰ با `GridSearchCV` (معیار Log-Loss) انتخاب کنید.
2. دقت و Log-Loss مدل منتخب را روی آزمون با مدل پایه مقایسه کنید.
3. نسبت شانس ویژگی‌ها (روی ویژگی‌های استانداردشده) را مرتب کنید. سه عامل مهم کدام‌اند؟

</div>

In [3]:
# Write your solution here

<div dir="rtl">

### پاسخ تمرین ب

</div>

In [4]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import log_loss

titanic = sns.load_dataset("titanic")
num = ["age", "fare", "sibsp", "parch", "pclass"]
cat = ["sex", "embarked"]
X_train, X_test, y_train, y_test = train_test_split(titanic[num + cat], titanic["survived"],
                                                    test_size=0.2, stratify=titanic["survived"],
                                                    random_state=42)
preprocess = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), num),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                          OneHotEncoder(handle_unknown="ignore")), cat)])
gs = GridSearchCV(make_pipeline(preprocess, LogisticRegression()),
                  {"logisticregression__C": [0.01, 0.1, 1, 10, 100]},
                  cv=StratifiedKFold(5, shuffle=True, random_state=42), scoring="neg_log_loss")
gs.fit(X_train, y_train)
for C, s in zip([0.01, 0.1, 1, 10, 100], gs.cv_results_["mean_test_score"]):
    print(f"C={C:<6} CV log-loss={-s:.3f}")
base = DummyClassifier(strategy="prior").fit(X_train, y_train)
print(f"best C={gs.best_params_['logisticregression__C']}  "
      f"test accuracy={gs.best_estimator_.score(X_test, y_test):.3f}  "
      f"test log-loss={log_loss(y_test, gs.predict_proba(X_test)[:, 1]):.3f}")
print(f"baseline test accuracy={base.score(X_test, y_test):.3f}  "
      f"log-loss={log_loss(y_test, base.predict_proba(X_test)[:, 1]):.3f}")

model = gs.best_estimator_
names = model[0].get_feature_names_out()
odds = pd.Series(np.exp(model[-1].coef_[0]), index=names).sort_values()
print(odds.round(3))

C=0.01   CV log-loss=0.521
C=0.1    CV log-loss=0.450
C=1      CV log-loss=0.445
C=10     CV log-loss=0.445
C=100    CV log-loss=0.445
best C=1  test accuracy=0.804  test log-loss=0.463
baseline test accuracy=0.615  log-loss=0.667
cat__sex_male      0.267
num__pclass        0.398
num__age           0.605
cat__embarked_S    0.699
num__sibsp         0.771
num__parch         0.936
cat__embarked_C    1.033
num__fare          1.105
cat__embarked_Q    1.356
cat__sex_female    3.667
dtype: float64


<div dir="rtl">

### پاسخ‌های کلیدی ب

- Log-Loss اعتبارسنجی برای C برابر ۰.۰۱، ۰.۱، ۱، ۱۰ و ۱۰۰ به ترتیب ۰.۵۲۱، ۰.۴۵۰، ۰.۴۴۵، ۰.۴۴۵ و ۰.۴۴۵ است. C بسیار کوچک کم‌برازش می‌کند و از C برابر ۱ به بعد تفاوتی نیست؛ C برابر **۱** انتخاب می‌شود.
- روی آزمون: دقت **۰.۸۰۴** و Log-Loss **۰.۴۶۳**، در برابر دقت ۰.۶۱۵ و Log-Loss ۰.۶۶۷ مدل پایه.
- نسبت شانس‌ها: زن بودن ۳.۶۷ و مرد بودن ۰.۲۷؛ کلاس سفر ۰.۴۰ (یک انحراف معیار کلاس بالاتر یعنی بلیت ارزان‌تر)؛ سن ۰.۶۱ برای هر انحراف معیار. سه عامل اصلی: **جنسیت**، **کلاس سفر** و **سن**.

</div>